#### Importing essential libraries and loading data files

In [1]:
import glob
from importlib.metadata import files
import duckdb
import pandas as pd
import pyreadstat

con = duckdb.connect("nsch_database.duckdb")

files_2024 = {
    "nsch_topical_2024":  "nsch/topical_2024/*.dta",
    "nsch_screener_2024": "nsch/screener_2024/*.dta",
}


files_2025 = {
    "nsch_topical_2025":  "nsch/topical_2025/*.dta",
    "nsch_screener_2024": "nsch/screener_2024/*.dta",
}


for table, pattern in files_2025.items():
    path = glob.glob(pattern)[0]
    df, meta = pyreadstat.read_dta(path, apply_value_formats=False)

    con.register("df", df)
    con.execute(f"CREATE OR REPLACE TABLE {table} AS SELECT * FROM df")
    con.unregister("df")

    # Variable labels (column descriptions)
    var_labels = pd.DataFrame({
        "column_name": list(meta.column_names),
        "label": list(meta.column_labels),
    })
    con.register("vl", var_labels)
    con.execute(f"CREATE OR REPLACE TABLE {table}_var_labels AS SELECT * FROM vl")
    con.unregister("vl")

    # Value labels (code -> text), e.g. 1 -> 'Yes'
    rows = []
    for var, mapping in meta.variable_value_labels.items():
        for code, text in mapping.items():
            try:
                rows.append((var, float(code), str(text)))
            except (TypeError, ValueError):
                pass
    val_labels = pd.DataFrame(rows, columns=["column_name", "code", "label"])
    con.register("vv", val_labels)
    con.execute(f"CREATE OR REPLACE TABLE {table}_value_labels AS SELECT * FROM vv")
    con.unregister("vv")

    print(table, df.shape)

con.close()

nsch_topical_2025 (45139, 458)
nsch_screener_2024 (116498, 38)


In [2]:
"""Load NSCH 2025 Stata files into DuckDB, with variable and value labels
parsed from the Census-provided .do files.

Run from the folder that contains nsch/topical_2024 and nsch/screener_2024.
Close any DuckDB CLI session and restart the notebook kernel first, so the
database file is not locked.
"""
import glob
import os
import re

import duckdb
import pandas as pd
import pyreadstat

DB = "nsch_database.duckdb"
FOLDERS = {
    "nsch_topical_2025": "nsch/topical_2025",
    "nsch_screener_2024": "nsch/screener_2024",
}

# Optional 'cap' / 'capture' / 'quietly' prefix, as in:
#   cap label values a1_active a1_active_lab
PRE = r'^\s*(?:(?:cap|capture|quietly|qui)\s+)?'
DEF_RE = re.compile(PRE + r'lab(?:el)?\s+def(?:ine)?\s+(\w+)\s+(.*)$', re.I)
VAL_RE = re.compile(PRE + r'lab(?:el)?\s+val(?:ues)?\s+(.*)$', re.I)
VAR_RE = re.compile(PRE + r'lab(?:el)?\s+var(?:iable)?\s+(\w+)\s+"([^"]*)"', re.I)
PAIR_RE = re.compile(r'(-?\d+|\.[A-Za-z]?)\s+"([^"]*)"')
OPT_RE = re.compile(r',\s*(?:add|modify|replace|nofix)\s*$', re.I)


def logical_lines(path):
    """Join Stata /// continuation lines and skip comment lines."""
    out, buf = [], ""
    with open(path, encoding="latin-1") as f:
        for raw in f:
            line = raw.rstrip("\r\n")
            if line.lstrip().startswith(("*", "//", "/*")):
                continue
            if line.rstrip().endswith("///"):
                buf += line.rstrip()[:-3] + " "
                continue
            out.append(buf + line)
            buf = ""
    if buf:
        out.append(buf)
    return out


def parse_do(path):
    """Return (label sets, variable -> label set, variable -> description)."""
    defs, assign, var_labels = {}, {}, {}
    for line in logical_lines(path):
        m = DEF_RE.match(line)
        if m:
            name = m.group(1)
            rest = OPT_RE.sub("", m.group(2).rstrip())
            defs.setdefault(name, {}).update(dict(PAIR_RE.findall(rest)))
            continue
        m = VAL_RE.match(line)
        if m:
            parts = m.group(1).split(",")[0].split()
            if len(parts) >= 2:
                for var in parts[:-1]:
                    assign[var.lower()] = parts[-1]
            continue
        m = VAR_RE.match(line)
        if m:
            var_labels[m.group(1).lower()] = m.group(2)
    return defs, assign, var_labels


def value_label_rows(defs, assign, actual):
    """Rows of (column_name, code, code_text, label).

    Special missing codes (.m, .n, .l, .d) have code = NULL and the
    text of the code in code_text.
    """
    rows = []
    for var, set_name in assign.items():
        col = actual.get(var)
        if col is None:
            continue
        for code, text in defs.get(set_name, {}).items():
            if code.startswith("."):
                rows.append((col, None, code, text))
            else:
                n = int(code)
                rows.append((col, float(n), str(n), text))
    return rows


def save(con, name, df):
    con.register("tmp_df", df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM tmp_df")
    con.unregister("tmp_df")


con = duckdb.connect(DB)

for table, folder in FOLDERS.items():
    dta_files = glob.glob(os.path.join(folder, "*.dta"))
    do_files = glob.glob(os.path.join(folder, "*.do"))
    if not dta_files:
        print(f"{table}: no .dta file in {folder}, skipping")
        continue

    df, meta = pyreadstat.read_dta(dta_files[0], apply_value_formats=False)
    actual = {c.lower(): c for c in df.columns}  # keep the data's own casing

    if do_files:
        defs, assign, do_var_labels = parse_do(do_files[0])
    else:
        defs, assign, do_var_labels = {}, {}, {}

    save(con, table, df)

    # Variable descriptions: prefer the .do file, fall back to the .dta
    var_rows = [
        (name, do_var_labels.get(name.lower()) or dta_label or "")
        for name, dta_label in zip(meta.column_names, meta.column_labels)
    ]
    save(con, f"{table}_var_labels",
         pd.DataFrame(var_rows, columns=["column_name", "label"]))

    # Value labels (code -> text)
    rows = value_label_rows(defs, assign, actual)
    save(con, f"{table}_value_labels",
         pd.DataFrame(rows, columns=["column_name", "code", "code_text", "label"]))

    n_labeled_vars = len({r[0] for r in rows})
    missing_in_data = sorted(set(assign) - set(actual))
    print(f"{table}: data {df.shape}, "
          f"{sum(1 for _, l in var_rows if l)} variable labels, "
          f"{len(rows)} value-label rows covering {n_labeled_vars} variables")
    if missing_in_data:
        print(f"  note: {len(missing_in_data)} labeled variables not in data "
              f"(e.g. {missing_in_data[:5]})")

con.close()

nsch_topical_2025: data (45139, 458), 458 variable labels, 3132 value-label rows covering 457 variables
nsch_screener_2024: data (116498, 38), 38 variable labels, 271 value-label rows covering 37 variables
